# Unit 20: Concurrent Execution Rings and Asynchronous Ingestion Topologies

## Speeding Up Workflows with Parallelization

Welcome back! You have mastered sequential workflows with prompt chaining and conditional paths via intelligent routing routers. Now, it is time to unlock major performance optimizations by learning **Parallel Processing**—executing multiple independent Claude API calls concurrently rather than waiting for each transaction to block the thread.

In this lesson, you will discover how to transform workflows that take minutes into non-blocking operations that complete in seconds. You will analyze the differences between synchronous and asynchronous architectures, master Python's native `asyncio` engine, and construct a multi-agent retrieval system that fires concurrent queries to Claude simultaneously.

---

## The Parallelization Workflow Pattern

Before writing asynchronous code, let's break down the high-level operational pattern. This architecture uses a clear two-phase execution lifecycle to achieve both ultra-low latency and data synthesis:

```text
 PHASE 1: PARALLEL RETRIEVAL                  PHASE 2: SEQUENTIAL SYNTHESIS
 
 ┌───────────────────────────┐
 │ Ingest Research Prompts   │
 └─────────────┬─────────────┘
               │
      ┌────────┼────────┐ (Concurrent Fan-out)
      ▼        ▼        ▼
   ┌────┐   ┌────┐   ┌────┐
   │QA-1│   │QA-2│   │QA-3│                     ┌───────────────────────────┐
   └─┬──┘   └─┬──┘   └─┬──┘                     │ Ingest Aggregated Datasets│
     │        │        │                        └─────────────┬─────────────┘
     └────────┼────────┘ (Awaited Consolidation)               │
              ▼                                                ▼
 ┌───────────────────────────┐                  ┌───────────────────────────┐
 │ Accumulate Array Results  │ ───────────────► │  Claude Synthesis Engine  │
 └───────────────────────────┘                  └─────────────┬─────────────┘
                                                              │
                                                              ▼
                                                ┌───────────────────────────┐
                                                │ Unified Production Guide  │
                                                └───────────────────────────┘

```

### Phase 1: Parallel Research Gathering

* **Concurrent Fan-Out:** The application launches multiple independent Claude API requests into the network loop at the exact same time.
* **Target Isolation:** Each individual worker thread researches an isolated aspect of your core domain topic (e.g., attractions, transport logistics, or cultural nuances).
* **Latency Optimization:** Because all connections execute concurrently, the entire batch processing window drops down to match the latency of the single slowest network request ($T_{\text{total}} \approx \max(t_1, t_2, \dots, t_n)$), rather than stacking them linearly ($T_{\text{total}} = \sum t_n$).
* **Order Preservation:** Collected text responses are consolidated and structured cleanly inside an indexed array matching your original prompt sequence.

### Phase 2: Sequential Result Synthesis

* **Upstream Ingestion:** The system collapses the parallel results array into a single structured markdown context block.
* **Contextual Blending:** This merged data is dispatched to a final Claude instance configured with explicit synthesis and layout instructions.
* **Unified Output:** The engine generates a cohesive document (like an integrated travel itinerary). This step guarantees that separate data points are cross-examined and written with a unified narrative tone.

This combination maximizes both execution speed and text consistency: you get the performance benefits of asynchronous networking for background research while maintaining tight quality control through sequential context consolidation. It is an exceptionally effective design pattern for large-scale data sweeps, multi-factor risk assessments, and complex analytical aggregations.

---

## Synchronous vs. Asynchronous Anthropic SDK Clients

The Anthropic SDK exposes two distinct client architectures to govern your request lifecycles. Choosing between them dictates whether your system thread freezes while waiting for a server response, or whether it drops back into an event loop to spawn adjacent workers.

### 1. The Standard Synchronous Client (`Anthropic`)

With the standard client instance, every message creation call acts as a blocking I/O barrier. The system freezes execution until the HTTP socket returns a complete response string.

```python
from anthropic import Anthropic

# Initialize the blocking synchronous client
client = Anthropic()

# Sequential Execution - Each block must wait for the preceding socket to close
response1 = client.messages.create(...)  # Thread blocks here until completion
response2 = client.messages.create(...)  # Begins execution only after response1 returns

```

### 2. The Asynchronous Client (`AsyncAnthropic`)

The async client relies on non-blocking sockets. Instead of freezing the thread during network latency, it yields control back to Python's central event coordinator, enabling adjacent calls to fire down the same pipe.

```python
from anthropic import AsyncAnthropic

# Initialize the non-blocking async client
client = AsyncAnthropic()

# Asynchronous Execution - Concurrent queries spawn inside an active coroutine loop
response1 = await client.messages.create(...)  # Dispatches and yields execution control
response2 = await client.messages.create(...)  # Dispatches immediately without waiting for response1

```

### Core Architecture Selector:

* **Deploy `Anthropic` (Sync):** For linear prompt chains, step-by-step verification guardrails, or quick scripts where downstream operations rely heavily on upstream variables.
* **Deploy `AsyncAnthropic` (Async):** For high-throughput batch operations, real-world multi-agent routers, and dynamic fan-out systems where performance optimization is critical.

---

## AsyncIO Fundamentals for Claude Workflows

Python’s native `asyncio` library provides a high-performance event loop that handles cooperative multitasking across I/O boundaries. The `async` keyword transforms a standard function definition into an evaluable **Coroutine** blueprint, while the `await` statement marks a non-blocking pause point—telling the event loop it can run other pending tasks while waiting for an external network response.

```python
import asyncio
from anthropic import AsyncAnthropic

# Instantiate our async gateway client
client = AsyncAnthropic()

async def fetch_domain_summary(question: str) -> str:
    # The 'await' operator suspends this coroutine, letting adjacent workers use the thread
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text

```

This asynchronous approach is highly effective for I/O-bound processes. Since LLM inference passes spend most of their execution lifecycle waiting on remote server processing and network transit, async functions ensure your local CPU never sits idle.

---

## Executing Asynchronous Runtimes via `asyncio.run()`

You cannot invoke an async coroutine directly from a standard synchronous runtime thread. Instead, you use `asyncio.run()` to spin up a managed event loop, execute the target entrypoint coroutine, and handle complete teardown operations cleanly upon exit.

```python
async def main():
    # Awaiting our async execution wrapper safely inside the loop environment
    result = await fetch_domain_summary("What are the top 3 must-see attractions in Paris?")
    print(result)

# Appending the standard bootstrap execution gate
if __name__ == "__main__":
    # Bootstraps the central loop architecture automatically
    asyncio.run(main())

```

---

## Concurrent Ingestion via `asyncio.gather()`

The true performance benefits of asynchronous architectures are realized when deploying `asyncio.gather()`. This utility takes an array of coroutine objects, maps them into concurrent tasks, executes them simultaneously, and returns a consolidated array containing all resolved values in their **exact original order**.

```python
async def main():
    # Instantiate an array of coroutine objects (Note: These have not started running yet)
    tasks = [
        fetch_domain_summary("What are the top attractions in Paris?"),
        fetch_domain_summary("How do I get around Paris?"),
        fetch_domain_summary("What are French cultural norms?")
    ]
    
    # Unpack tasks and execute them concurrently across the network
    # The results array maps perfectly to the input sequence index: [attr, transport, culture]
    results = await asyncio.gather(*tasks)

```

While worker `A` waits for its first inference token to travel back across the network, the underlying event loop switches contexts instantly to fire worker `B` and worker `C`. This turns what used to be stacked, idle waiting time into concurrent execution space.

---

## Building a Production-Ready Parallel Research Architecture

Let's combine these asynchronous primitives into an operational travel synthesis engine. We will write an explicit worker coroutine that returns named tuples, build out an array of research targets, and run a final synthesis aggregation pass.

### 1. Structuring the Asynchronous Worker Coroutine

```python
from anthropic import AsyncAnthropic

# Core client instantiation
client = AsyncAnthropic()

async def ask_question(question: str) -> tuple[str, str]:
    """
    Dispatches a non-blocking inference query to Claude and tracks execution states.
    """
    print(f"🔄 Spawning Worker: {question}")
    
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system="You are an expert travel researcher. Provide high-density, concise data points.",
        messages=[{"role": "user", "content": question}]
    )
    
    answer = response.content[0].text
    print(f"✅ Worker Completed: {question}")
    
    # Returning a tuple ensures that downstream aggregation filters trace securely
    return question, answer

```

---

### 2. Defining Independent Research Context Arrays

We select target queries that can be processed entirely in parallel without any cross-dependent variables.

```python
# Our targeted independent research matrix
questions = [
    "What are the top 3 must-see attractions in Paris?",
    "What is the most efficient way to get around Paris as a tourist?",
    "What are important cultural etiquette tips for visitors to France?"
]

```

---

### 3. Implementing the Sequential Aggregation Engine

Once the concurrent data sweep clears, we wrap the results inside a unified context template and pass it to a final sequential processing step to synthesize our data.

```python
async def create_travel_plan(results: list[tuple[str, str]]) -> str:
    """
    Ingests raw parallel research arrays and synthesizes them into a final guide.
    """
    combined_research = ""
    for question, answer in results:
        combined_research += f"### Research Prompt: {question}\n{answer}\n\n"
    
    # Constructing a unified synthesis directive
    aggregator_prompt = (
        f"Analyze and compile the following raw background research into a polished, "
        f"cohesive Paris travel guide:\n\n{combined_research}\n\n"
        f"Enforce standard markdown formatting, ensure sections flow logically, and remove redundant points."
    )
    
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4000,
        system="You are a senior editorial travel architect. Synthesize raw details into professional blueprints.",
        messages=[{"role": "user", "content": aggregator_prompt}]
    )
    
    return response.content[0].text

```

---

### 4. Running the Complete Multi-Agent Parallel Loop

We wire our ingestion tasks, data gather layers, and synthesis engines straight into the central execution runtime:

```python
import asyncio

async def main():
    print(f"Initializing Async Matrix: Deploying {len(questions)} Parallel Research Workers...")
    
    # Programmatically compile our collection of task coroutines
    tasks = [ask_question(q) for q in questions]
    
    # Fire all research queries concurrently across the network
    results = await asyncio.gather(*tasks)
    print("\n🎉 Parallel Data Sweep Completed! Launching Synthesis Phase...")
    
    # Route the consolidated arrays into our sequential aggregator block
    final_travel_guide = await create_travel_plan(results)
    
    print("\n================ FINAL COMPILED PRODUCTION GUIDE ================")
    print(final_travel_guide)

if __name__ == "__main__":
    asyncio.run(main())

```

---

## Technical Trace Analysis

When executing this asynchronous file, your system console logs display a distinct three-phase operational timeline:

```text
Starting 3 research questions in parallel...
🔄 Spawning Worker: What are the top 3 must-see attractions in Paris?
🔄 Spawning Worker: What is the most efficient way to get around Paris as a tourist?
🔄 Spawning Worker: What are important cultural etiquette tips for visitors to France?
✅ Worker Completed: What is the most efficient way to get around Paris as a tourist?
✅ Worker Completed: What are important cultural etiquette tips for visitors to France?
✅ Worker Completed: What are the top 3 must-see attractions in Paris?

Research completed!

```

### Deconstructing the Trace:

1. **Simultaneous Fan-Out:** All three `🔄 Spawning Worker` diagnostic logs flash onto the terminal screen **at the exact same instant**. This verifies that your requests are firing concurrently rather than queuing up sequentially.
2. **Asynchronous Out-of-Order Resolution:** Notice that the `✅ Worker Completed` notifications can return in a completely different order than they were dispatched (e.g., transport resolving before attractions). This proves that the underlying event loop is processing tasks cooperatively, optimizing for whichever socket finishes its work first.
3. **Deterministic Final Synthesis:** The final output merges all three disparate components into a clean, well-formatted guide, showing how parallel speed can be beautifully paired with high-quality sequential analysis.

---

## Production Use Cases and Optimization Boundaries

Evaluating when to apply parallel processing chains is critical for building cost-effective, maintainable software architectures.

| Optimal Concurrent Use Cases | Structural Benefits |
| --- | --- |
| **Comprehensive Field Research** | Pulls from diverse, independent topic sectors simultaneously to build broad knowledge repositories rapidly. |
| **Batch Analytical Processing** | Evaluates large arrays of user interaction transcripts, database text blobs, or files without freezing your UI. |
| **Competitive Scenario Matrixing** | Queries separate instances using unique personas simultaneously to compare cross-domain risks side-by-side. |

---

## Conclusion

You have mastered the configuration patterns needed to translate slow, blocking code structures into high-speed asynchronous execution networks. By combining non-blocking parallel research gathering with a final sequential synthesis phase, you ensure your pipelines run at maximum velocity without losing any contextual precision.

Let's dive into the practical code labs to stand up, optimize, and test your first high-throughput concurrent agent mesh!

## Converting Sync to Async Claude Calls

Time to take your first step from theory into practice by converting a working synchronous Claude program into an asynchronous one! You've learned about the power of async programming, and now you'll experience the fundamental transformation that makes parallel workflows possible.

Your starter code contains a simple program with an ask_question function that queries Claude using the standard synchronous client. Your mission is to convert it to use async patterns by following the TODO comments strategically placed throughout the code.

Here's what you need to do:

    Change the import from Anthropic to AsyncAnthropic.
    Update the client initialization to use AsyncAnthropic instead of Anthropic.
    Transform the ask_question function into an async function using the async keyword.
    Add the await keyword before the API call to handle the asynchronous response.
    Update the main function to use await when calling ask_question.
    Transform the main function into an async function using the async keyword.
    Update the program execution to use asyncio.run() instead of calling main() directly.

The program should work exactly the same way when you're done, but it will now be ready to handle the parallel processing patterns you'll build in upcoming exercises. This conversion is the essential foundation that unlocks Claude's true speed potential!

To transform your synchronous API code into a non-blocking asynchronous architecture, you need to replace the standard connection client, declare your execution layers as coroutines using `async`, and mark your network boundaries with the `await` keyword.

Here is the complete, converted Python script with all your `asyncio` and `AsyncAnthropic` integration points fully implemented:

```python
import asyncio
# Import AsyncAnthropic instead of the standard synchronous client
from anthropic import AsyncAnthropic

# Initialize the non-blocking async Anthropic client
client = AsyncAnthropic()

# Transform ask_question into an async function using the 'async' keyword
async def ask_question(question):
    """
    Async function to ask Claude a single question
    """
    print(f"🔄 Asking: {question}")
    
    # Add the 'await' keyword before the API call to handle the asynchronous response safely
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system="You are a travel expert. Give brief, helpful answers.",
        messages=[{"role": "user", "content": question}]
    )
    
    answer = response.content[0].text
    print(f"✅ Answered: {question}")
    
    return question, answer

# Transform the main function into an async coroutine using the 'async' keyword
async def main():
    # Define a single test research question
    question = "What are the top 3 must-see attractions in Paris?"
    
    # Update the call to use the 'await' keyword when invoking ask_question
    q, a = await ask_question(question)
    
    print(f"\nResult for '{q}':")
    print(a)

# Update the program execution block to use asyncio.run() instead of calling main() directly
if __name__ == "__main__":
    # Bootstraps the central event loop to handle your async code execution
    asyncio.run(main())

```

### Why This Refactoring Changes the Game:

* **Non-Blocking Infrastructure:** By converting `ask_question` into an async function, you turn its network call into an awaitable task. When your program hits `await client.messages.create(...)`, the execution thread releases its lock, allowing it to manage other background tasks instead of freezing.
* **Managed Event Loop Entry:** Since an async function cannot run on a naked, synchronous main thread, utilizing `asyncio.run(main())` provisions the engine's cooperative event loop automatically upon file execution. This ensures that your workspace is fully prepared for the advanced parallel `asyncio.gather()` configurations in the next exercises! 🚀

## Building a Parallel Claude Workflow

Great job mastering async calls! Now you're ready for the next level: transforming your sequential async code into a parallel powerhouse.

The given code uses async properly, but it's still processing questions one at a time in a for loop. Your goal is to unleash true parallel processing by making all those Claude calls run simultaneously!

Here's what you need to do:

    Replace the sequential for loop with parallel task creation using a list comprehension.
    Use asyncio.gather() to execute all tasks simultaneously.

When you're done, all your "🔄 Asking" messages will appear instantly, followed by "✅ Answer received!" messages as responses arrive. This shows your requests are running concurrently instead of waiting for each other!

```python
import asyncio
from anthropic import AsyncAnthropic

# Initialize the async Anthropic client
client = AsyncAnthropic()

async def ask_question(question):
    print(f"🔄 Asking: {question}")
    
    # Send async request to Claude
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system="You are a travel expert. Give brief, helpful answers.",
        messages=[{"role": "user", "content": question}]
    )
    
    # Extract and return the answer
    answer = response.content[0].text
    print(f"✅ Answer received!")
    
    return answer

async def main():
    # List of independent questions for Paris trip planning
    questions = [
        "What are the top 3 must-see attractions in Paris?",
        "What is the most efficient way to get around Paris as a tourist?",
        "What are important cultural etiquette tips for visitors to France?"
    ]
    
    # TODO: Create tasks for all questions using a list comprehension
    
    # TODO: Execute the tasks in parallel using the asyncio gather method with the unpacking operator
    results = []
    for question in questions:
        answer = await ask_question(question)
        results.append(answer)
    
    # Display all results
    for i, answer in enumerate(results):
        print(f"\nAnswer {i+1}: {answer}")

# Run the async function
if __name__ == "__main__":
    asyncio.run(main())
```

To break free from the sequential blocking pattern of the `for` loop and unlock true concurrency, you need to separate **task creation** from **task execution**.

By using a list comprehension, you instantly instantiate all your coroutine objects. Passing them into `asyncio.gather(*tasks)` using the unpacking operator (`*`) instructs Python's event loop to fire all network requests simultaneously.

Here is your updated script with the parallel generation matrix fully implemented:

```python
import asyncio
from anthropic import AsyncAnthropic

# Initialize the async Anthropic client
client = AsyncAnthropic()

async def ask_question(question):
    print(f"🔄 Asking: {question}")
    
    # Send async request to Claude
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system="You are a travel expert. Give brief, helpful answers.",
        messages=[{"role": "user", "content": question}]
    )
    
    # Extract and return the answer
    answer = response.content[0].text
    print(f"✅ Answer received!")
    
    return answer

async def main():
    # List of independent questions for Paris trip planning
    questions = [
        "What are the top 3 must-see attractions in Paris?",
        "What is the most efficient way to get around Paris as a tourist?",
        "What are important cultural etiquette tips for visitors to France?"
    ]
    
    print(f"Starting {len(questions)} research questions in parallel...\n")
    
    # Programmatically batch and instantiate all your coroutine tasks simultaneously
    tasks = [ask_question(question) for question in questions]
    
    # Unpack and fire all tasks concurrently across the event loop
    results = await asyncio.gather(*tasks)
    
    print("\n🎉 All parallel research complete!")
    
    # Display all results (results are preserved in the original order of the input list)
    for i, answer in enumerate(results):
        print(f"\nAnswer {i+1}: {answer}")

# Run the async function
if __name__ == "__main__":
    asyncio.run(main())

```

### What to Look For in Your Terminal Trace:

When you run this updated script, you will notice a complete shift in execution behavior:

1. **The Instant Fan-Out:** All three `🔄 Asking:...` notifications will burst onto your screen **at the exact same fraction of a second**.
2. **Concurrent Resolution:** The `✅ Answer received!` updates will print out as they finish. Because the event loop optimizes for whichever network socket finishes processing first, these can resolve out of order depending on server latency.
3. **Index Stability:** Despite resolving out of order over the wire, `asyncio.gather()` takes care of re-sorting the outputs, ensuring your final `results` list perfectly mirrors the sequence of your original `questions` array! 🚀

## Adding Result Synthesis to Parallel Workflows

Excellent work mastering parallel execution! Now it's time to complete the two-phase workflow pattern by adding the crucial aggregation step that transforms your scattered research into a unified, actionable result.

Your code currently runs multiple Claude calls in parallel and displays each answer separately. While this demonstrates the power of concurrent processing, the real magic happens when you synthesize all that parallel research into one comprehensive guide that is more valuable than the sum of its parts.

Here's what you need to accomplish:

    Create an async function that:
        Combines all question-answer pairs into a formatted research string.
        Send this combined research to Claude with instructions to create a brief, practical travel guide.
        Returns the travel guide.
    Update your main function to call this aggregation step after the parallel research completes.
    Display the final travel guide created by Claude.

This final step transforms your lightning-fast parallel workflow into a complete system that delivers both speed and intelligence!

```python
import asyncio
from anthropic import AsyncAnthropic

# Initialize the async Anthropic client
client = AsyncAnthropic()

# List of independent questions for Paris trip planning
questions = [
    "What are the top 3 must-see attractions in Paris?",
    "What is the most efficient way to get around Paris as a tourist?",
    "What are important cultural etiquette tips for visitors to France?"
]


async def ask_question(question):
    """
    Async function to ask Claude a single question
    """
    print(f"🔄 Asking: {question}")
    
    # Send async request to Claude
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system="You are a travel expert. Give brief, helpful answers.",
        messages=[{"role": "user", "content": question}]
    )
    
    # Extract the answer
    answer = response.content[0].text
    
    print(f"✅ Answered: {question}")
    
    return question, answer


# Create an async function that takes the research results as a parameter
    # Combine all Q&A pairs into a formatted research string
    # Send the combined research to Claude with instructions to create a brief, practical travel guide
    # Return the travel guide


async def main():
    """
    Execute parallel research and create comprehensive travel plan
    """
    print(f"Starting {len(questions)} research questions in parallel...")
    
    # Create tasks for all questions
    tasks = [ask_question(question) for question in questions]
    
    # Execute all tasks in parallel
    results = await asyncio.gather(*tasks)
    
    print("\nResearch completed!")
    
    # TODO: Call your aggregation function with the results and store the return value
    
    # TODO: Print the travel plan with a nice header like "Paris Travel Guide:"

# Run the parallel workflow
if __name__ == "__main__":
    asyncio.run(main())
```